## Denoising Diffusion Probability Models (DDPM)


In [ ]:
import math

import tqdm

from einops import rearrange


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# from datasets import load_dataset

# ds = load_dataset("ILSVRC/imagenet-1k", streaming=True, trust_remote_code=True)

In [ ]:

# fig, subplots = plt.subplots(4, 4, squeeze=False, figsize=(15,15))
# subplots = subplots.flatten()

# for i,x in enumerate(ds["train"]):
#     im = torch.tensor(np.array(x["image"]))
#     im = rearrange(im, "h w c -> 1 c h w")
#     print(im.shape)
#     new_shape = (256, 256)
#     im = F.interpolate(im, new_shape, mode="bilinear")
#     im = rearrange(im, "1 c h w -> h w c")
#     subplots[i].imshow(im)

#     if i == 15:
#         break

## Model

In [ ]:

class TimeEmbedding(nn.Module):

    def __init__(self, d, max_period=10000.0):
        super().__init__()
        
        self.d = d
        half = d // 2
        self.max_period = max_period
        thetas = torch.exp(-math.log(self.max_period) * torch.arange(0, half, dtype=torch.float32) / half)

        self.register_buffer('thetas', thetas)
    
    def forward(self, t):

        if t.dim() == 0:

            res = torch.zeros(self.d)
            res[0::2] = torch.cos(self.thetas * t)
            res[1::2] = torch.sin(self.thetas * t)

        else:
            res = torch.zeros(t.shape[0], self.d)

            # t is unsqueezed to size (batch, 1), since thetas is of size (d,), t is broadcast in the last dimension to (batch, d).
            res[:, 0::2] = torch.cos(self.thetas * t.unsqueeze(-1))
            res[:, 1::2] = torch.sin(self.thetas * t.unsqueeze(-1))
        
        
        return res.to(t.device).to(dtype=t.dtype)



class ResBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels

        self.norm1 = nn.BatchNorm2d(num_features=in_channels)
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1)
        self.act1 = nn.ReLU(inplace=True)

        self.pe = TimeEmbedding(out_channels)
        self.pe_head = nn.Linear(out_channels, out_channels * 2)
        
        self.norm2 = nn.BatchNorm2d(num_features=out_channels)
        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1)
        self.act2 = nn.ReLU(inplace=True)

        self.skip_proj = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        
    
    def forward(self, x, t):

        skip = self.skip_proj(x)
        x = self.norm1(x)
        x = self.conv1(x)
        x = self.act1(x)

        pos_enc = self.pe(t)
        if pos_enc.dim() == 1:
            pos_enc = pos_enc.unsqueeze(0)
        
        pos_enc = self.pe_head(pos_enc)
        gamma, beta = torch.split(pos_enc, self.out_channels, dim=-1)
        
        B = x.shape[0]
        C = self.out_channels
        gamma = gamma.view(B, C, 1, 1)
        beta = beta.view(B, C, 1, 1)

        x = x * (1 + gamma) + beta

        x = self.norm2(x)
        x = self.conv2(x)
        x = self.act2(x)
        return x + skip


class CropAndConcat(nn.Module):

    def forward(self, x: torch.Tensor, contracting_x: torch.Tensor):

        contracting_x = torchvision.transforms.functional.center_crop(contracting_x, (x.shape[2], x.shape[3]))
        x = torch.cat([x, contracting_x], dim=1)

        return x

class Upsample(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up_conv = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2
        )

    def forward(self, x):
        return self.up_conv(x)


class UNet(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.down_conv = nn.ModuleList([ResBlock(i, o) for (i, o) in
                                        [(in_channels, 64), (64, 128), (128, 256), (256, 512)]])
        self.down_sample = nn.ModuleList([nn.MaxPool2d(2) for _ in range(4)])

        self.middle_conv = ResBlock(512, 1024)

        self.up_sample = nn.ModuleList([Upsample(i, o) for (i, o) in 
                                        [(1024, 512), (512, 256), (256, 128), (128, 64)]])
        self.up_conv = nn.ModuleList([ResBlock(i, o) for (i, o) in
                                        [(1024, 512), (512, 256), (256, 128), (128, 64)]])

        self.crop_and_concat = CropAndConcat()
        
        self.final_conv = nn.Conv2d(64, out_channels=out_channels, kernel_size=1)

    def forward(self, x, t):

        pass_through = []
        for i,conv in enumerate(self.down_conv):

            x = conv(x, t)
            pass_through.append(x)
            x = self.down_sample[i](x)
        
        x = self.middle_conv(x, t)

        for i,up in enumerate(self.up_sample):

            x = up(x)
            x = self.crop_and_concat(x, pass_through.pop())
            x = self.up_conv[i](x, t)
        
        x = self.final_conv(x)
        return x

## Training

In [ ]:
class DDPMDataset:

    def __init__(self, batch=32, batches_per_epoch=6250):
        self.batch_size = batch
        self.batches_per_epoch = batches_per_epoch  # Define epoch length

        # dataset = "ILSVRC/imagenet-1k"
        dataset = "nielsr/CelebA-faces"
        self.ds = load_dataset(dataset, streaming=True, trust_remote_code=True, split="train").shuffle(seed=42, buffer_size=1000).batch(batch)
        # For inference (no augmentation)
        self.process = transforms.Compose([
            transforms.Lambda(lambda img: img.convert('RGB')),  # Convert to RGB (3 channels)
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

        self.total_steps = 100
        self.final_variance = 1.

        self.beta = 2.0 / self.total_steps 
        self.alpha = 1.0 - self.beta

        self.iterator = None
        self.current_batch = 0
        
    def __len__(self):
        return self.batches_per_epoch
    
    def __iter__(self):
        self.current_batch = 0
        return self

    def __next__(self):
        if self.current_batch >= self.batches_per_epoch:
            raise StopIteration
            
        # Create iterator on first call or when exhausted
        if self.iterator is None:
            self.iterator = iter(self.ds)
        
        try:
            x = next(self.iterator)
        except StopIteration:
            # Reset iterator when dataset is exhausted (infinite cycling)
            self.iterator = iter(self.ds)
            x = next(self.iterator)
            
        self.current_batch += 1
        imgs = torch.stack([self.process(img) for img in x["image"]], dim=0)
        batch_size, channels, height, width = imgs.shape


        beta = self.beta
        times = torch.randint(1, self.total_steps + 1, (batch_size,), dtype=torch.float32)
        alpha_bar = self.alpha ** times

        scale = torch.sqrt(alpha_bar)
        noise_variance = 1 - alpha_bar

        eps = torch.randn((batch_size, channels, height, width))
        noise = eps * torch.sqrt(noise_variance).view(batch_size, 1, 1, 1)

        return imgs * scale.view(batch_size, 1, 1, 1) + noise, times, eps, imgs


fig, plots = plt.subplots(3)
test = DDPMDataset(batch=32)

for x in test:
    
    noisy = (x[0][0].permute(1, 2, 0).numpy() + 1) / 2
    times = x[1]
    noise = (x[2][0].permute(1, 2, 0).numpy() + 1) / 2
    img = (x[3][0].permute(1, 2, 0).numpy() + 1) / 2


    plots[0].imshow(noisy)
    plots[1].imshow(noise)
    plots[2].imshow(img)

    break

In [ ]:

def train_model(model, dataset, epochs=10, lr=1e-4,
                criterion=None, optimizer=None, scheduler=None,
                device=None, batch_log=100):

    # Select device automatically if not provided
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Set defaults if not explicitly provided
    optimizer = optimizer or optim.Adam(model.parameters(), lr=lr)
    criterion = criterion
    scheduler = scheduler or optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

    train_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        total_batches = len(dataset)
        pbar = tqdm.tqdm(enumerate(dataset), total=total_batches,
                        desc=f"Epoch {epoch+1}/{epochs}")
        
        # Training loop
        # We ignore the actual image from the dataset
        for batch_idx, (inputs, times, targets, images) in pbar:
            
            inputs, times, targets = inputs.to(device), times.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs, times)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

            if (batch_idx + 1) % batch_log == 0:
                avg_loss = epoch_loss / (batch_idx + 1)
                pbar.set_postfix({'Training Loss': f"{avg_loss:.4f}"})

        avg_epoch_loss = epoch_loss / (batch_idx + 1)
        train_losses.append(avg_epoch_loss)
        print(f"Epoch {epoch+1} Training Loss: {avg_epoch_loss:.4f}")

    return {
        'model': model,
        'train_losses': train_losses,
    }


In [ ]:

import gc

torch.cuda.empty_cache()
gc.collect()


In [ ]:

dataset = DDPMDataset()
model = UNet(3, 3)

train_model(model, dataset, criterion=nn.MSELoss())

In [ ]:

torch.save(model.state_dict(), "ddpm_model.pt")
print("Saved ddpm_model.pt")


## Inference

In [ ]:


model.load_state_dict(torch.load("ddpm_model.pt"))
model = model.to("cuda")
num_generated = 5


beta = 2.0 / 100.
sigma = math.sqrt(beta)
times = torch.ones((num_generated,), dtype=torch.float32, device="cuda") * 100
generated = torch.randn((num_generated, 3, 256, 256), device="cuda")

with torch.no_grad():
    for t in reversed(range(1,101)):

        denoise = model.forward(generated, times)
        times -= 1.0
        
        alpha = 1.0 - beta
        alpha_bar = alpha ** t

        generated = 1 / math.sqrt(alpha) * ( generated - denoise * (1.0 - alpha) / math.sqrt(1.0 - alpha_bar) )

        if t != 1:
            noise = torch.randn((num_generated, 3, 256, 256), device="cuda") * sigma
            generated += noise
        print(t)

to_pil = transforms.ToPILImage()
generated_vis = (generated.detach().cpu().clamp(-1, 1) + 1) / 2
images = [to_pil(generated_vis[i]) for i in range(num_generated)]

fig, plots = plt.subplots(num_generated, figsize=(9, 3 * num_generated))

for i, x in enumerate(images):
    plots[i].imshow(x)


In [ ]:
# Visualize 50-step noising and denoising on 5 images (same denoising as previous cell)

# Config
num_samples = 5
T_partial = 50
beta = 2.0 / 100.0  # same schedule as training/inference (constant beta, T=100)
alpha = 1.0 - beta
sigma = math.sqrt(beta)

# Device and model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()

# Get 5 images from the streaming dataset and preprocess like training
# Assumes `dataset` is an instance of ImageNetDataset defined above
hf_batch = next(iter(dataset.ds))
imgs0 = torch.stack([dataset.process(img) for img in hf_batch['image'][:num_samples]], dim=0).to(device)

# Forward diffusion to t = 50 (x_t = sqrt(ᾱ_t) x0 + sqrt(1-ᾱ_t) ε)
with torch.no_grad():
    t = T_partial
    alpha_bar_t = alpha ** t
    scale = math.sqrt(alpha_bar_t)
    noise_std = math.sqrt(1.0 - alpha_bar_t)

    eps = torch.randn_like(imgs0)
    x_t = scale * imgs0 + noise_std * eps

    # Reverse process from t -> 0 using the same update rule and noise as the previous cell
    times = torch.full((num_samples,), float(t), dtype=torch.float32, device=device)
    x = x_t.clone()
    for cur_t in reversed(range(1, T_partial + 1)):
        denoise = model(x, times)  # predicts ε
        times -= 1.0

        alpha_bar_cur = alpha ** cur_t
        # x_{t-1} = 1/sqrt(alpha) * (x_t - ((1-alpha)/sqrt(1-ᾱ_t)) * ε̂)
        x = (x - denoise * (1.0 - alpha) / math.sqrt(1.0 - alpha_bar_cur)) / math.sqrt(alpha)

        if cur_t != 1:
            noise = torch.randn_like(x) * sigma
            x = x + noise

    x0_hat = x

# To PIL for display (map from [-1, 1] to [0, 1])
to_pil = transforms.ToPILImage()

def to_vis(tensor_bchw):
    return ((tensor_bchw.detach().clamp(-1, 1) + 1.0) / 2.0).cpu()

orig_vis = to_vis(imgs0)
noisy_vis = to_vis(x_t)
denoi_vis = to_vis(x0_hat)

# Plot
fig, axes = plt.subplots(num_samples, 3, figsize=(9, 3 * num_samples))
if num_samples == 1:
    axes = [axes]

for i in range(num_samples):
    axes[i][0].imshow(to_pil(orig_vis[i]))
    axes[i][0].set_title('Original')
    axes[i][0].axis('off')

    axes[i][1].imshow(to_pil(noisy_vis[i]))
    axes[i][1].set_title(f'Noisy (t={T_partial})')
    axes[i][1].axis('off')

    axes[i][2].imshow(to_pil(denoi_vis[i]))
    axes[i][2].set_title('Denoised')
    axes[i][2].axis('off')

plt.tight_layout()
plt.show()